In [ ]:
!pip install -U sentence-transformers datasets

from google.colab import drive
from IPython.display import clear_output
drive.mount('/content/drive')

clear_output()

In [2]:
import json
import logging
import os

import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from transformers import TrainerCallback
from collections import defaultdict
from sklearn.metrics import (
    f1_score, accuracy_score, recall_score, precision_score
)
import torch
import numpy as np
import pandas as pd

os.environ["WANDB_MODE"] = "disabled"
logging.basicConfig(level=logging.INFO)

model_name = "hfl/chinese-roberta-wwm-ext"
output_dir = "/content/drive/MyDrive/roberta-contrastive-model"

train_batch_size = 16
max_seq_length = 128

num_train_epochs = 10

print("🚀 Using device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))

model = SentenceTransformer(model_name)
model.max_seq_length = max_seq_length

with open("training_data.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

samples = []
for pair in raw_data:
    s1 = pair["song1"]["lyrics"]
    s2 = pair["song2"]["lyrics"]
    label = float(pair["label"])
    samples.append(InputExample(texts=[s1, s2], label=label))

split_idx = int(0.8 * len(samples))
train_samples = samples[:split_idx]
val_samples = samples[split_idx:]

train_dataset = Dataset.from_list([
    {"sentence1": s.texts[0], "sentence2": s.texts[1], "label": s.label}
    for s in train_samples
])

eval_dataset = Dataset.from_list([
    {"sentence1": s.texts[0], "sentence2": s.texts[1], "label": s.label}
    for s in val_samples
])

print(f"Train pairs: {len(train_dataset)}")
print(f"Val pairs:   {len(eval_dataset)}")

loss_fn = losses.CosineSimilarityLoss(model)

train_loss_history = []
eval_loss_history = []

class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Step {state.global_step} | {logs}")
            if "loss" in logs:
                train_loss_history.append((state.global_step, logs["loss"]))
            if "eval_loss" in logs:
                eval_loss_history.append((state.global_step, logs["eval_loss"]))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average="macro")
    return {
        "f1_macro": f1
    }

training_args = SentenceTransformerTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,

    per_device_train_batch_size=train_batch_size,
    warmup_ratio=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    metric_for_best_model="eval_f1_macro",
    load_best_model_at_end=True,
    greater_is_better=True,

    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to=[]
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss_fn,
    callbacks=[PrintLossCallback()],
)

trainer.train()

model.save(output_dir)
print(f"Final model saved to: {output_dir}")

🚀 Using device: cuda
CUDA device name: NVIDIA A100-SXM4-80GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/412M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Train pairs: 512
Val pairs:   128


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss
1,0.249600,0.238807
2,0.185000,0.236612
3,0.106000,0.239348


Step 10 | {'loss': 0.3364, 'grad_norm': 1.8619240522384644, 'learning_rate': 4.5e-05, 'epoch': 0.3125}
Step 20 | {'loss': 0.242, 'grad_norm': 2.5211501121520996, 'learning_rate': 4.476744186046512e-05, 'epoch': 0.625}
Step 30 | {'loss': 0.2496, 'grad_norm': 2.564908981323242, 'learning_rate': 3.895348837209303e-05, 'epoch': 0.9375}
Step 32 | {'eval_loss': 0.2388067990541458, 'eval_runtime': 0.539, 'eval_samples_per_second': 237.474, 'eval_steps_per_second': 29.684, 'epoch': 1.0}
Step 40 | {'loss': 0.1664, 'grad_norm': 2.6873281002044678, 'learning_rate': 3.313953488372093e-05, 'epoch': 1.25}
Step 50 | {'loss': 0.191, 'grad_norm': 2.6815340518951416, 'learning_rate': 2.7325581395348836e-05, 'epoch': 1.5625}
Step 60 | {'loss': 0.185, 'grad_norm': 1.9057486057281494, 'learning_rate': 2.1511627906976744e-05, 'epoch': 1.875}
Step 64 | {'eval_loss': 0.236612007021904, 'eval_runtime': 0.5055, 'eval_samples_per_second': 253.238, 'eval_steps_per_second': 31.655, 'epoch': 2.0}
Step 70 | {'loss':

In [3]:
model_path = "/content/drive/MyDrive/roberta-contrastive-model"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_path, device=device)

def test_results(file):
  with open(file, "r", encoding="utf-8") as f:
    data = json.load(f)

  genre_mode_groups = defaultdict(list)
  threshold = 0.5

  for pair in data:
    g1 = pair["song1"]["genre"]
    g2 = pair["song2"]["genre"]
    label = int(pair["label"])
    s1 = pair["song1"]["lyrics"]
    s2 = pair["song2"]["lyrics"]

    all_genres = set(g1 + g2)

    for genre in all_genres:
        in_both = genre in g1 and genre in g2
        mode = "per-genre" if in_both else "cross-genre"
        genre_mode_groups[(genre, mode)].append((s1, s2, label))

  rows = []

  for (genre, mode), group in genre_mode_groups.items():
    sents1 = [x[0] for x in group]
    sents2 = [x[1] for x in group]
    labels = [x[2] for x in group]

    emb1 = model.encode(sents1, convert_to_tensor=True, device=device)
    emb2 = model.encode(sents2, convert_to_tensor=True, device=device)
    cos_sim = util.cos_sim(emb1, emb2).diagonal().cpu().numpy()
    preds = (cos_sim > threshold).astype(int)

    row = {
        "Genre": genre,
        "Mode": mode,
        "Accuracy": accuracy_score(labels, preds),
        "F1 Micro": f1_score(labels, preds, average="micro"),
        "F1 Weighted": f1_score(labels, preds, average="weighted"),
        "F1 Macro": f1_score(labels, preds, average="macro"),
        "Recall": recall_score(labels, preds),
        "Precision": precision_score(labels, preds),
    }
    rows.append(row)

  mode_summary = []

  for mode in ["per-genre", "cross-genre"]:
    all_s1, all_s2, all_labels = [], [], []
    for (genre_key, mode_key), group in genre_mode_groups.items():
        if mode_key == mode:
            all_s1 += [x[0] for x in group]
            all_s2 += [x[1] for x in group]
            all_labels += [x[2] for x in group]

    if all_labels:
        emb1 = model.encode(all_s1, convert_to_tensor=True, device=device)
        emb2 = model.encode(all_s2, convert_to_tensor=True, device=device)
        cos_sim = util.cos_sim(emb1, emb2).diagonal().cpu().numpy()
        preds = (cos_sim > threshold).astype(int)

        row = {
            "Mode": mode,
            "Accuracy": accuracy_score(all_labels, preds),
            "F1 Micro": f1_score(all_labels, preds, average="micro"),
            "F1 Weighted": f1_score(all_labels, preds, average="weighted"),
            "F1 Macro": f1_score(all_labels, preds, average="macro"),
            "Recall": recall_score(all_labels, preds),
            "Precision": precision_score(all_labels, preds),
        }
        mode_summary.append(row)

  df = pd.DataFrame(rows)
  mode_df = pd.DataFrame(mode_summary)

  pd.set_option("display.max_columns", None)

  print("Metrics per Genre:")
  print(df.sort_values(["Genre", "Mode"]).to_string(index=False))

  print("\nMetrics per Mode:")
  print(mode_df.sort_values("Mode").to_string(index=False))

In [4]:
test_results("./testing_data_1.json")

Metrics per Genre:
Genre        Mode  Accuracy  F1 Micro  F1 Weighted  F1 Macro   Recall  Precision
民俗与传统 cross-genre  0.666667  0.666667     0.673046  0.641148 0.700000   0.777778
民俗与传统   per-genre  1.000000  1.000000     1.000000  1.000000 1.000000   1.000000
 爱与浪漫 cross-genre  0.615385  0.615385     0.613863  0.614370 0.550000   0.647059
 爱与浪漫   per-genre  0.733333  0.733333     0.733333  0.732143 0.714286   0.714286
生活与反思 cross-genre  0.670732  0.670732     0.667624  0.668315 0.571429   0.727273
生活与反思   per-genre  0.461538  0.461538     0.455012  0.448485 0.333333   0.400000
社会与现实 cross-genre  0.750000  0.750000     0.758333  0.733333 0.800000   0.571429
社会与现实   per-genre  0.750000  0.750000     0.750000  0.666667 0.500000   0.500000
风景与旅程 cross-genre  0.689655  0.689655     0.689655  0.689655 0.666667   0.714286
风景与旅程   per-genre  0.862069  0.862069     0.858802  0.847368 0.944444   0.850000

Metrics per Mode:
       Mode  Accuracy  F1 Micro  F1 Weighted  F1 Macro   Recall  Precis

In [ ]:
test_results("./testing_data_2.json")